### Transformaciones simples: select, filter/where, agregaciones y joins

In [25]:
# importamos librerias
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, StringType, DoubleType, StructType, StructField

In [26]:
# creamos sesion spark
spark = (SparkSession.builder
         .appName("aplicacion")
         .master("local[*]")
         .config("spark.driver.memory", "2g")
         .getOrCreate()
)

In [27]:
# creamos un dataset persona, compra
persona = spark.createDataFrame([
    ("pedro", "españa", 6, 1),
    ("luis", "francia", 43, 2),
    ("alejandro", "italia", 53, 3),
    ("jose", "españa", 24, 4),
    ("carlos", "francia", 45, 5)],
    ("nombre", "pais", "edad", "ID")
)
persona.show()

+---------+-------+----+---+
|   nombre|   pais|edad| ID|
+---------+-------+----+---+
|    pedro| españa|   6|  1|
|     luis|francia|  43|  2|
|alejandro| italia|  53|  3|
|     jose| españa|  24|  4|
|   carlos|francia|  45|  5|
+---------+-------+----+---+



In [28]:
compra = spark.createDataFrame([
    ("2", "play", 400),
    ("1", "fifa", 50),
    ("4", "movil", 800),
    ("3", "zapatillas", 80),
    ("4", "camiseta", 35)],
    ("id_persona","nombre","precio")
)
compra.show()

+----------+----------+------+
|id_persona|    nombre|precio|
+----------+----------+------+
|         2|      play|   400|
|         1|      fifa|    50|
|         4|     movil|   800|
|         3|zapatillas|    80|
|         4|  camiseta|    35|
+----------+----------+------+



In [29]:
# probamos el select
listado_select_1= ["nombre", "pais"]
df_select_1 = persona.select(listado_select_1).show()

+---------+-------+
|   nombre|   pais|
+---------+-------+
|    pedro| españa|
|     luis|francia|
|alejandro| italia|
|     jose| españa|
|   carlos|francia|
+---------+-------+



In [30]:
# aplicamos filtros
df_filtro1 = persona.filter(F.col("edad") > 40).select("nombre", "pais", "edad").show()

+---------+-------+----+
|   nombre|   pais|edad|
+---------+-------+----+
|     luis|francia|  43|
|alejandro| italia|  53|
|   carlos|francia|  45|
+---------+-------+----+



In [31]:
# uso withcolumn
df_withcolumn = persona.withColumn("edad_x_id", F.col("edad") * F.col("id")).show()

+---------+-------+----+---+---------+
|   nombre|   pais|edad| ID|edad_x_id|
+---------+-------+----+---+---------+
|    pedro| españa|   6|  1|        6|
|     luis|francia|  43|  2|       86|
|alejandro| italia|  53|  3|      159|
|     jose| españa|  24|  4|       96|
|   carlos|francia|  45|  5|      225|
+---------+-------+----+---+---------+



In [32]:
# filtro con where
persona.where("pais == 'españa'").show()

+------+------+----+---+
|nombre|  pais|edad| ID|
+------+------+----+---+
| pedro|españa|   6|  1|
|  jose|españa|  24|  4|
+------+------+----+---+



In [33]:
# agregaciones: group + agg
persona.groupBy(F.col("pais")).agg(
    F.avg(F.col("edad")).alias("media_edad"),
    F.count("pais").alias("cantidad")
).show()

+-------+----------+--------+
|   pais|media_edad|cantidad|
+-------+----------+--------+
| españa|      15.0|       2|
|francia|      44.0|       2|
| italia|      53.0|       1|
+-------+----------+--------+



In [34]:
# practicamos los joins simples
persona.join(compra, on= (F.col("id") == F.col("id_persona")), how="inner").show()

+---------+-------+----+---+----------+----------+------+
|   nombre|   pais|edad| ID|id_persona|    nombre|precio|
+---------+-------+----+---+----------+----------+------+
|    pedro| españa|   6|  1|         1|      fifa|    50|
|     luis|francia|  43|  2|         2|      play|   400|
|alejandro| italia|  53|  3|         3|zapatillas|    80|
|     jose| españa|  24|  4|         4|     movil|   800|
|     jose| españa|  24|  4|         4|  camiseta|    35|
+---------+-------+----+---+----------+----------+------+



In [35]:
spark.stop()